# 🤖 Altron Mobile AI Assistant - Google Colab Android APK Builder

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/global-linguists-ai/Altron/blob/master/Altron_Android_Build_Colab.ipynb)

This official notebook allows you to build the standalone **Altron Android APK** directly in Google Colab using cloud CPU/GPU VMs without needing a high-end local development machine.

---

### Step 1: Clone or Download Altron Project Code
Clones the repository directly from GitHub or downloads the packaged source bundle.

In [ ]:
!rm -rf /content/altron
# Try cloning repository first; fallback to direct package download
!git clone https://github.com/global-linguists-ai/Altron.git /content/altron || \
 (mkdir -p /content/altron && \
  wget -O /content/altron.zip https://ais-dev-uwtyadws3migd6n4zhm7ic-145044719806.asia-southeast1.run.app/altron-android.zip && \
  unzip -q -o /content/altron.zip -d /content/altron)

%cd /content/altron
!ls -la

### Step 2: Install Node.js Dependencies
Installs React Native and Expo dependencies using npm.

In [ ]:
%cd /content/altron
!npm install --legacy-peer-deps

### Step 3: Install Java 17 & Android SDK in Colab VM
Sets up OpenJDK 17, Android command-line tools, Android SDK 35, and accepts licenses automatically.

In [ ]:
# Install OpenJDK 17
!apt-get update -qq && apt-get install -y -qq openjdk-17-jdk wget unzip

import os
os.environ['JAVA_HOME'] = '/usr/lib/jvm/java-17-openjdk-amd64'
os.environ['ANDROID_HOME'] = '/content/android-sdk'
os.environ['PATH'] += ':/content/android-sdk/cmdline-tools/latest/bin:/content/android-sdk/platform-tools'

# Setup Android SDK Command-line Tools
!mkdir -p /content/android-sdk/cmdline-tools
!wget -q -nc https://dl.google.com/android/repository/commandlinetools-linux-11076708_latest.zip -O /content/cmdline-tools.zip
!unzip -q -o /content/cmdline-tools.zip -d /content/android-sdk/cmdline-tools
!rm -rf /content/android-sdk/cmdline-tools/latest
!mv /content/android-sdk/cmdline-tools/cmdline-tools /content/android-sdk/cmdline-tools/latest

# Accept Android Licenses & Install platforms
!yes | /content/android-sdk/cmdline-tools/latest/bin/sdkmanager --licenses > /dev/null
!/content/android-sdk/cmdline-tools/latest/bin/sdkmanager "platform-tools" "platforms;android-35" "build-tools;35.0.0" > /dev/null

# Set local.properties sdk.dir
!echo "sdk.dir=/content/android-sdk" > /content/altron/android/local.properties
!java -version

### Step 4: Build Standalone Android APK with Gradle
Runs the Gradle build to compile the standalone APK.

In [ ]:
%cd /content/altron/android
!chmod +x ./gradlew
# Build Release or Debug APK
!./gradlew assembleRelease --no-daemon -x lint || ./gradlew assembleDebug --no-daemon

### Step 5: Download the Generated APK
Triggers automatic browser download of the generated `.apk` file.

In [ ]:
from google.colab import files
import glob

apks = glob.glob('/content/altron/android/app/build/outputs/apk/**/*.apk', recursive=True)
if apks:
    print('✅ Generated APK(s) found:')
    for apk in apks:
        print(' ->', apk)
    # Download the first APK found (prefer release or debug)
    target_apk = apks[0]
    for apk in apks:
        if 'release' in apk:
            target_apk = apk
            break
    print(f'Downloading {target_apk} to your machine...')
    files.download(target_apk)
else:
    print('❌ No APK found. Please review Step 4 gradle build output.')